# Día 1 · Visualizar siniestros de tráfico en Barcelona

**Pregunta del módulo.** ¿En qué zonas y franjas horarias conviene revisar primero los expedientes registrados?

Vamos a cargar el CSV del curso, comprobar qué estamos contando y construir gráficos con pandas y Seaborn. Usaremos Matplotlib para el anillo, las barras apiladas y el área apilada. Al final hay una demostración opcional de cómo preparar un resumen con Spark para dibujarlo en Python.

**Ruta para clase.** Empieza por barras, histograma, línea, dispersión, barras al 100 %, mapa de calor y mapa. Los demás gráficos forman una galería que puedes intercalar según el tiempo. Las celdas muestran ejemplos completos.

## 0 · Abrir el notebook y preparar el entorno

- **Jupyter o VS Code.** Abre el cuaderno desde el repositorio y selecciona un kernel de Python. El CSV está en `PySpark/data`.
- **Colab.** Abre el notebook y sube el CSV con el botón del panel **Archivos**. Mantén el nombre `2023_accidents_causa_conductor_gu_bcn_.csv`.
- Si falta alguna biblioteca, ejecuta en una celda `%pip install pandas "seaborn>=0.13" matplotlib folium`. Los gráficos de Seaborn funcionan sin conexión una vez instaladas las bibliotecas y descargado el CSV. El mapa geográfico usa Folium y necesita conexión para cargar las calles de OpenStreetMap.

## 1 · Cargar los datos y decidir qué cuenta como un siniestro

El archivo contiene registros de causas vinculados a expedientes. Un expediente puede ocupar varias filas. Para los gráficos de siniestros dejamos **una fila por expediente** después de comprobar que las filas repetidas coinciden en fecha, hora y ubicación. Guardamos el archivo original en `registros` por si queremos analizar las causas por separado.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

archivo = '2023_accidents_causa_conductor_gu_bcn_.csv'
inicio = Path.cwd()
carpetas = [inicio, inicio / 'data', inicio.parent / 'data', inicio / 'PySpark' / 'data']
ruta = next((carpeta / archivo for carpeta in carpetas if (carpeta / archivo).is_file()), None)
if ruta is None:
    raise FileNotFoundError(f'Sube {archivo} al entorno o abre el notebook desde el repositorio.')

registros = pd.read_csv(ruta)
registros[['Numero_expedient', 'Nom_districte', 'Hora_dia', 'Descripcio_causa_mediata']].head()

In [ ]:
campos = {
    'Numero_expedient': 'expediente',
    'Nom_districte': 'distrito',
    'Nom_barri': 'barrio',
    'Descripcio_torn': 'turno',
    'NK_Any': 'year',
    'Mes_any': 'month',
    'Dia_mes': 'day',
    'Hora_dia': 'hora',
    'Latitud_WGS84': 'latitud',
    'Longitud_WGS84': 'longitud',
}
tabla = registros[list(campos)].rename(columns=campos).copy()
tabla['expediente'] = tabla['expediente'].str.strip().replace('', pd.NA)
print(f"Filas de origen: {len(tabla):,}")
print(f"Filas sin expediente: {tabla['expediente'].isna().sum():,}")
tabla = tabla.dropna(subset=['expediente'])

# Antes de quedarnos con una fila, comprobamos que las demás dicen lo mismo.
conflictos = tabla.groupby('expediente').nunique(dropna=False).gt(1).any(axis=1)
if conflictos.any():
    raise ValueError(f'Hay {conflictos.sum()} expedientes con datos contradictorios. Revísalos antes de agrupar.')

siniestros = tabla.drop_duplicates(subset='expediente').copy()
siniestros['fecha'] = pd.to_datetime(siniestros[['year', 'month', 'day']])
siniestros['distrito'] = siniestros['distrito'].replace({'Desconegut': 'Desconocido'})
siniestros['barrio'] = siniestros['barrio'].replace({'Desconegut': 'Desconocido'})
siniestros['turno'] = siniestros['turno'].replace({'Matí': 'Mañana', 'Tarda': 'Tarde', 'Nit': 'Noche'})

dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
turnos = ['Mañana', 'Tarde', 'Noche']
siniestros['dia_semana'] = siniestros['fecha'].dt.dayofweek.map(dict(enumerate(dias)))
siniestros['mes'] = siniestros['fecha'].dt.to_period('M').dt.to_timestamp()

print(f"Expedientes distintos: {len(siniestros):,}")
print(f"Expedientes con distrito desconocido: {siniestros['distrito'].eq('Desconocido').sum():,}")
siniestros[['distrito', 'barrio', 'fecha', 'hora', 'turno']].head()

**Para comentar.** El CSV tiene 8.897 filas y 7.721 expedientes. El número de filas puede servir para describir el archivo. Para describir el número de siniestros usamos expedientes distintos. La categoría «Desconocido» se mantiene visible para poder valorar la cobertura territorial.

In [ ]:
# Tablas pequeñas para alimentar los gráficos.
por_distrito = (siniestros.groupby('distrito').size().rename('expedientes')
                .sort_values(ascending=False).reset_index())
por_turno = siniestros.groupby('turno').size().reindex(turnos, fill_value=0)
por_turno = por_turno.rename('expedientes').reset_index()
por_turno['porcentaje'] = 100 * por_turno['expedientes'] / len(siniestros)

# Los días sin expedientes se representan como cero dentro del año cubierto.
calendario = pd.date_range('2023-01-01', '2023-12-31', freq='D', name='fecha')
por_dia = (siniestros.groupby('fecha').size().reindex(calendario, fill_value=0)
           .rename('expedientes').reset_index())
por_dia['dia_semana'] = por_dia['fecha'].dt.dayofweek.map(dict(enumerate(dias)))
por_mes = (por_dia.groupby(pd.Grouper(key='fecha', freq='MS'))['expedientes']
           .sum().reset_index())

colores_turno = {'Mañana': '#067276', 'Tarde': '#613A89', 'Noche': '#B65C15'}

## 2 · Una variable categórica o numérica

Las **barras** comparan categorías. El **histograma** muestra cómo se distribuye una medida numérica. El **diagrama de caja** resume esa distribución. En los dos últimos gráficos la unidad pasa a ser el **día**, con su total de expedientes.

### 01 · Barras horizontales | una variable categórica

¿Qué distritos tienen más expedientes registrados? Con Seaborn usamos `data` para elegir la tabla y `x` e `y` para indicar sus columnas. Una barra horizontal facilita leer nombres largos. Las barras muestran volumen registrado. Para comparar el riesgo necesitaríamos un denominador, por ejemplo el volumen de tráfico.

In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=por_distrito, x='expedientes', y='distrito', color='#613A89')
ax.set(title='Expedientes registrados por distrito', xlabel='Expedientes', ylabel='Distrito')
plt.show()

### 02 · Histograma | una variable numérica

¿Cuántos expedientes suele haber en un día? El eje horizontal agrupa totales diarios en intervalos. La altura indica cuántos días hay en cada intervalo.

In [ ]:
ax = sns.histplot(data=por_dia, x='expedientes', bins=18, color='#067276')
ax.set(title='Distribución de los totales diarios', xlabel='Expedientes por día', ylabel='Número de días')
plt.show()

### 03 · Diagrama de caja | resumen de una distribución

La línea dentro de la caja es la mediana. La caja contiene la mitad central de los días. Los puntos que quedan fuera merecen revisión y pueden ser válidos.

In [ ]:
ax = sns.boxplot(data=por_dia, x='expedientes', color='#067276')
ax.set(title='Mediana y dispersión de los totales diarios', xlabel='Expedientes por día')
plt.show()

## 3 · Proporciones y denominadores

Una proporción divide una parte por un total. Escribe siempre de qué total hablas. Aquí cada expediente tiene un solo turno. Si contáramos categorías de causas, un mismo expediente podría estar en varias y los porcentajes podrían sumar más del 100 %.

### 04 · Barras de porcentaje | comparación precisa

¿Qué porcentaje del total corresponde a cada turno? Las barras permiten comparar diferencias con facilidad. Los turnos pueden tener duraciones distintas.

In [ ]:
ax = sns.barplot(data=por_turno, x='turno', y='porcentaje', hue='turno',
                 order=turnos, palette=colores_turno, legend=False)
ax.set(title='Porcentaje de expedientes por turno', xlabel='Turno', ylabel='% del total anual')
for barras in ax.containers:
    ax.bar_label(barras, fmt='%.1f%%', padding=3)
ax.set_ylim(0, 100)
plt.show()

### 05 · Anillo | partes de un total con pocas categorías

Muestra los mismos datos como partes de un total. Es útil para explicar la composición general con pocas categorías. Compara con las barras si quieres distinguir porcentajes cercanos.

In [ ]:
fig, ax = plt.subplots()
ax.pie(por_turno['expedientes'], labels=por_turno['turno'],
       colors=[colores_turno[turno] for turno in por_turno['turno']],
       autopct='%.1f%%', wedgeprops={'width': 0.55})
ax.set_title('Cómo se reparte el total anual por turno')
plt.show()

## 4 · Dos variables

Con una fecha y una medida podemos ver evolución. Con una categoría y una medida podemos comparar distribuciones. Con dos medidas numéricas podemos explorar si cambian juntas.

### 06 · Línea temporal | fecha y medida numérica

¿Cómo cambia el recuento de un mes a otro? Ordenamos el eje por fecha. Los meses tienen distinta duración. Si eso importa para la comparación, podemos calcular expedientes por día.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(data=por_mes, x='fecha', y='expedientes', marker='o', ax=ax)
ax.set(title='Expedientes registrados por mes en 2023', xlabel='Mes', ylabel='Expedientes')
fig.autofmt_xdate()
plt.show()

### 07 · Cajas por categoría | día de la semana y total diario

¿Son parecidos los totales diarios de lunes y sábados? Una caja por día de la semana permite comparar la mediana y la dispersión entre categorías.

In [ ]:
plt.figure(figsize=(9, 4))
ax = sns.boxplot(data=por_dia, x='dia_semana', y='expedientes', order=dias, color='#067276')
ax.set(title='Distribución diaria por día de la semana', xlabel='Día de la semana', ylabel='Expedientes por día')
plt.show()

### 08 · Violín | forma de la distribución por categoría

Usa los mismos datos que el gráfico anterior. El ancho de cada violín muestra dónde se concentran los valores. El contorno es una estimación suavizada. En grupos pequeños conviene ver también los valores individuales.

In [ ]:
plt.figure(figsize=(9, 4))
ax = sns.violinplot(data=por_dia, x='dia_semana', y='expedientes',
                    order=dias, inner='box', cut=0, color='#067276')
ax.set(title='Forma de los totales diarios por día de la semana', xlabel='Día de la semana', ylabel='Expedientes por día')
plt.show()

### 09 · Dispersión | dos variables numéricas

¿Los días con más expedientes por la mañana también suelen tener más por la tarde? Cada punto es un día. La posición muestra sus dos recuentos. Días con la misma combinación pueden quedar superpuestos.

In [ ]:
por_dia_turno = (pd.crosstab(siniestros['fecha'], siniestros['turno'])
                 .reindex(index=calendario, columns=turnos, fill_value=0).reset_index())
por_dia_turno['tipo_dia'] = por_dia_turno['fecha'].dt.dayofweek.map(
    lambda dia: 'Fin de semana' if dia >= 5 else 'Entre semana'
)


ax = sns.scatterplot(data=por_dia_turno, x='Mañana', y='Tarde', alpha=0.55)
ax.set(title='Mañana y tarde en el mismo día', xlabel='Expedientes de mañana', ylabel='Expedientes de tarde')
plt.show()

## 5 · Tres variables y composición por grupo

Podemos añadir una variable usando color, paneles o tamaño. Primero leemos los ejes. Después comprobamos qué significa el color o el tamaño. Cada elemento visual tiene que responder a una parte de la pregunta.

### 10 · Dispersión con color | dos medidas y una categoría

Añadimos el tipo de día al gráfico anterior. El color separa los días entre semana de los fines de semana. Buscamos diferencias entre los patrones antes de interpretar la nube completa.

In [ ]:
ax = sns.scatterplot(data=por_dia_turno, x='Mañana', y='Tarde', hue='tipo_dia', alpha=0.65)
ax.set(title='Mañana y tarde según el tipo de día', xlabel='Expedientes de mañana', ylabel='Expedientes de tarde')
ax.legend(title='Tipo de día')
plt.show()

### 11 · Barras agrupadas | dos categorías y un recuento

¿Cuántos expedientes hubo en cada turno dentro de cada distrito? Las barras se colocan lado a lado para comparar cantidades.

In [ ]:
distrito_turno = (siniestros.groupby(['distrito', 'turno']).size()
                  .rename('expedientes').reset_index())

plt.figure(figsize=(10, 7))
ax = sns.barplot(data=distrito_turno, x='expedientes', y='distrito', hue='turno',
                 hue_order=turnos, palette=colores_turno, errorbar=None)
ax.set(title='Volumen de expedientes por distrito y turno', xlabel='Expedientes', ylabel='Distrito')
ax.legend(title='Turno')
plt.show()

### 12 · Barras apiladas al 100 % | comparar proporciones

Cada barra usa como denominador el total de su propio distrito. Dos distritos pueden tener proporciones parecidas y cantidades muy distintas. Consulta los recuentos en `distrito_turno` para dar contexto a los porcentajes.

In [ ]:
distrito_turno['porcentaje'] = (100 * distrito_turno['expedientes']
                               / distrito_turno.groupby('distrito')['expedientes'].transform('sum'))
proporciones = (distrito_turno.pivot(index='distrito', columns='turno', values='porcentaje')
                .reindex(columns=turnos).fillna(0))
ax = proporciones.plot.barh(stacked=True, color=[colores_turno[t] for t in turnos], figsize=(9, 6))
ax.set(title='Composición por turno en cada distrito', xlabel='% dentro del distrito', ylabel='Distrito', xlim=(0, 100))
ax.legend(title='Turno')
plt.show()

### 13 · Mapa de calor | día, hora y recuento

Cada celda cruza un día de la semana con una hora. El color representa cuántos expedientes se registraron en esa combinación. Es una forma rápida de localizar franjas para revisar.

In [ ]:
dia_hora = pd.crosstab(siniestros['dia_semana'], siniestros['hora'])
dia_hora = dia_hora.reindex(index=dias, columns=range(24), fill_value=0)

plt.figure(figsize=(11, 4))
ax = sns.heatmap(dia_hora, cmap='Blues', cbar_kws={'label': 'Expedientes'})
ax.set(title='Expedientes registrados por día y hora', xlabel='Hora del día', ylabel='Día de la semana')
plt.show()

### 14 · Área apilada | evolución y composición

La altura total muestra el volumen del mes. El grosor de cada banda muestra la aportación de un turno. La línea simple del ejemplo 06 suele ser más fácil de leer si solo interesa el total.

In [ ]:
mes_turno = (siniestros.groupby(['mes', 'turno']).size()
             .rename('expedientes').reset_index().sort_values('mes'))

series_turno = (mes_turno.pivot(index='mes', columns='turno', values='expedientes')
                .reindex(columns=turnos).fillna(0))
fig, ax = plt.subplots(figsize=(9, 4))
ax.stackplot(series_turno.index, *[series_turno[t] for t in turnos],
             labels=turnos, colors=[colores_turno[t] for t in turnos])
ax.set(title='Evolución mensual de los expedientes por turno', xlabel='Mes', ylabel='Expedientes')
ax.legend(title='Turno')
fig.autofmt_xdate()
plt.show()

### 15 · Paneles pequeños | un gráfico por turno

Separamos la serie temporal en tres paneles, uno por turno. Como comparten la escala vertical, es más fácil comparar tanto las cantidades como los cambios. Compáralos con el área apilada.

In [ ]:
paneles = sns.relplot(data=mes_turno, kind='line', x='mes', y='expedientes',
                     col='turno', col_order=turnos, marker='o', height=3, aspect=1.2)
paneles.set_axis_labels('Mes', 'Expedientes')
paneles.set_titles('{col_name}')
for ax in paneles.axes.flat:
    ax.tick_params(axis='x', labelrotation=35)
paneles.figure.suptitle('Evolución mensual en cada turno', y=1.03)
plt.show()

## 6 · Ubicación

Un mapa responde **dónde**. Aquí el punto representa una celda aproximada de coordenadas, y su tamaño indica cuántos expedientes quedaron dentro. El recuento observado puede reflejar que por una zona pasan más vehículos. Para comparar riesgo haría falta información de exposición.

### 16 · Mapa geográfico | latitud, longitud y recuento

Folium coloca los puntos sobre un mapa de Barcelona. Agrupamos las coordenadas en celdas de 0,005 grados para reducir las marcas. El tamaño del círculo representa el número de expedientes de esa celda. Al pasar el ratón verás su recuento. Puedes acercarte para ubicar las zonas respecto a las calles.

In [ ]:
import folium

ubicados = siniestros.dropna(subset=['latitud', 'longitud']).copy()
paso = 0.005
for columna in ['latitud', 'longitud']:
    ubicados[columna] = (ubicados[columna].div(paso).round().mul(paso)).round(3)

celdas_mapa = (ubicados.groupby(['latitud', 'longitud']).size()
               .rename('expedientes').reset_index())
print(f'Expedientes con coordenadas: {len(ubicados):,} de {len(siniestros):,}')
print(f'Burbujas del mapa: {len(celdas_mapa):,}')



mapa = folium.Map(location=[41.39, 2.16], zoom_start=12, tiles='OpenStreetMap')
for fila in celdas_mapa.itertuples(index=False):
    folium.CircleMarker(
        location=[fila.latitud, fila.longitud],
        radius=2 + fila.expedientes**0.5,
        color='#067276', fill=True, fill_opacity=0.65,
        tooltip=f'{fila.expedientes} expedientes en esta celda',
    ).add_to(mapa)

mapa

**Cómo leer el mapa.** Los círculos señalan el centro aproximado de cada celda. Son recuentos de expedientes registrados. Para comparar riesgo entre calles o zonas necesitaríamos también saber cuánto tráfico circula por ellas. [Mapas interactivos con Folium](https://python-visualization.github.io/folium/latest/user_guide/map.html)

## 7 · Opcional. Preparar el resumen con PySpark

La galería usa pandas para preparar las tablas y Seaborn para dibujarlas. En Big Data, el mismo patrón consiste en resumir en Spark y transferir a pandas **solo las filas que requiere el gráfico**. Aquí repetimos las barras por distrito contando expedientes distintos. Esta celda necesita el entorno de PySpark del curso. Puedes saltarla si no lo tienes preparado.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master('local[2]').appName('Siniestros por distrito').getOrCreate()
try:
    origen = spark.read.csv(ruta.resolve().as_uri(), header=True)
    resumen_spark = (origen.select(
        F.trim('Numero_expedient').alias('expediente'),
        F.when(F.col('Nom_districte') == 'Desconegut', 'Desconocido')
         .otherwise(F.col('Nom_districte')).alias('distrito'),
    ).filter(F.col('expediente').isNotNull() & (F.col('expediente') != ''))
     .groupBy('distrito').agg(F.countDistinct('expediente').alias('expedientes')))

    # Solo transferimos una fila por distrito, una vez hecha la agregación.
    para_grafico = pd.DataFrame([fila.asDict() for fila in resumen_spark.collect()])
finally:
    spark.stop()

assert (para_grafico.set_index('distrito')['expedientes'].sort_index().to_dict()
        == por_distrito.set_index('distrito')['expedientes'].sort_index().to_dict())

plt.figure(figsize=(9, 5))
ax = sns.barplot(data=para_grafico.sort_values('expedientes', ascending=False),
                 x='expedientes', y='distrito', color='#613A89')
ax.set(title='El mismo resumen calculado con Spark', xlabel='Expedientes', ylabel='Distrito')
plt.show()

## 8 · Ejercicio final. Elige el gráfico y defiende tu respuesta

Trabaja con `siniestros`, que contiene una fila por expediente. Responde con un gráfico y una frase que cite una cifra. Indica el periodo, las unidades y el denominador cuando uses porcentajes. Hay varias formas válidas de resolver cada pregunta.

1. **Comparar categorías.** ¿Cuáles son los cinco barrios con más expedientes registrados durante el primer semestre? Mantén aparte los registros con barrio desconocido e indica cuántos son.
2. **Describir una medida.** ¿Cómo se distribuye el número diario de expedientes los fines de semana? Señala un valor típico y explica qué harías si vieras un día muy alejado del resto.
3. **Ver un cambio en el tiempo.** ¿En qué meses se registraron más expedientes de noche? Compara el total mensual con el promedio diario del mes y comprueba si cambia la lectura.
4. **Comparar proporciones.** ¿Qué turno representa la mayor proporción de expedientes en Eixample y en Sant Martí? Indica el total que sirve de base a los porcentajes en cada distrito.
5. **Relacionar dos medidas.** ¿Cómo se relacionan los recuentos de tarde y noche en un mismo día? Añade una tercera variable para ver si la lectura cambia entre días de lunes a viernes y fines de semana.
6. **Cruzar categorías.** ¿Qué combinación de día de la semana y hora destacarías para revisar en Eixample durante el segundo semestre? Indica cuántos expedientes hay en la celda que señalas.
7. **Ubicar un patrón.** Representa en un mapa geográfico los expedientes del turno de noche. ¿Qué celdas revisarías primero? Explica qué dato adicional necesitarías para comparar el riesgo entre zonas.

Puedes repartir las preguntas entre grupos y poner las respuestas en común. Para cerrar, elige un resultado y escribe **qué expedientes revisarías a continuación y por qué**.

[Volver al README](../README.md) · [CSV del curso](../data/2023_accidents_causa_conductor_gu_bcn_.csv) · [Día 2: construir el panel](16_panel_siniestros_dia_2.ipynb) · [Guía de Seaborn](https://seaborn.pydata.org/tutorial/function_overview.html)

In [ ]:
# Añade una celda por pregunta.
# 1. Filtra las filas que responden a tu pregunta.
# 2. Prepara la tabla y dibuja el gráfico.
# 3. Escribe tu interpretación en una celda de texto.